# Proof of Concept
This cell is a POC to read all of the comments from a PEP pull request into a MongoDB collection. The rest of the notebook adapts this POC to scale for multiple Python Enhancement Proposals (PEPs), so PEPs, pull requests and comments are keyed together in a MongoDB document collection.

In [ ]:
import requests
from pymongo import MongoClient
import tiktoken
import json

import json

with open("config.json") as f:
    config = json.load(f)

gh_token = config["github_token"]
mongo_connection_string = config["mongo_uri"]

headers = {"Authorization": f"Bearer {gh_token}"}
tokenizer = tiktoken.get_encoding("cl100k_base")

client = MongoClient(mongo_connection_string)
db = client.consensus
collection = db.github_pr_comments

def fetch_pr_comments(owner, repo, pr_number):
    url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}/comments"
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return resp.json()

def store_comment(comment):
    text = comment["body"]
    tokens = tokenizer.encode(text)

    doc = {
        "source": "github",
        "url": comment["html_url"],
        "repo": f"{owner}/{repo}",
        "type": "pull_request_comment",
        "author": comment["user"]["login"],
        "created_at": comment["created_at"],
        "raw_text": text,
        "tokens": tokens,
        "metadata": {"comment_id": comment["id"]}
    }
    collection.insert_one(doc)

comments = fetch_pr_comments("openrobotics", "reps", 16)
for c in comments:
    store_comment(c)


Fetching submissions from python/typing-council...

Successfully extracted 58 Typing Council submissions.
Inserting/Updating records in MongoDB collection 'typing_council.submissions'...
MongoDB sync complete.
Saved local exports to Excel and CSV.
   Seq #  Issue #        Date                                                        Topic / Proposal Title                                           Vote Link                      Draft / Text Link (PR)
0      1        2  2023-11-28                             What are the subtyping rules for `tuple[T, ...]`?   https://github.com/python/typing-council/issues/2                          N/A (No PR linked)
1      2        3  2023-11-29                                          Adopt the initial Python typing spec   https://github.com/python/typing-council/issues/3  https://github.com/python/typing/pull/1517
2      3        5  2024-01-10        Allow `NoReturn` to appear outside of function return type annotations   https://github.com/python/typi

In [3]:
from datetime import datetime, timezone
import hashlib
import json
import re
import time
from pymongo import MongoClient, ReplaceOne
import requests

# 1. Load Configuration
with open("config.json") as f:
    config = json.load(f)

# 2. Setup MongoDB Connection
mongo_connection_string = config["mongo_uri"]
client = MongoClient(mongo_connection_string, maxPoolSize=10)
db = client.consensus

pep_prs_collection = db.pep_prs
pep_voting_summary_col = db.pep_voting_summary

# Salt for consistent pseudonymization across notebook cells
USER_SALT = config.get("user_salt", "pep_consensus_salt_2026")
PEPS_API_URL = "https://peps.python.org/api/peps.json"

# Cutoff Date: August 1, 2018
AUG_2018_CUTOFF = datetime(2018, 8, 1, tzinfo=timezone.utc)

# Regex patterns
SOFT_VOTE_PATTERN = re.compile(
    r"(?<![a-zA-Z0-9_])([\+\-](?:1|0|0\.5|1\/2|\.5))(?![a-zA-Z0-9_])"
)
DISCOURSE_TOPIC_REGEX = re.compile(r"discuss\.python\.org/t/(?:[^/]+/)?(\d+)")


def pseudonymize_username(username: str) -> str:
    """Generate a consistent SHA-256 pseudonymized identifier for a user."""
    if not username:
        return "anon_unknown"
    salted_str = f"{USER_SALT}:{username.strip().lower()}"
    return f"user_{hashlib.sha256(salted_str.encode('utf-8')).hexdigest()[:12]}"


def fetch_pep_index():
    """Fetch official PEP metadata index from peps.python.org/api/peps.json."""
    try:
        resp = requests.get(
            PEPS_API_URL,
            headers={"User-Agent": "PEP-Vote-Collector/1.0"},
            timeout=15,
        )
        if resp.ok:
            return resp.json()
    except Exception as e:
        print(f"Error fetching PEP API index: {e}")
    return {}


def parse_date(date_str):
    """Parse various ISO and PEP date formats safely."""
    if not date_str:
        return None
    for fmt in ("%Y-%m-%d", "%d-%b-%Y", "%Y/%m/%d"):
        try:
            dt = datetime.strptime(str(date_str).strip(), fmt)
            return dt.replace(tzinfo=timezone.utc)
        except ValueError:
            pass
    return None


def extract_discourse_topic_ids(pep_meta):
    """Extract topic IDs from Resolution, Post-History, and Discourse-Topic fields."""
    topic_ids = set()

    target_fields = []
    if pep_meta.get("resolution"):
        target_fields.append(str(pep_meta["resolution"]))
    if pep_meta.get("discourse_topic"):
        target_fields.append(str(pep_meta["discourse_topic"]))

    post_history = pep_meta.get("post_history")
    if isinstance(post_history, list):
        target_fields.extend([str(item) for item in post_history])
    elif isinstance(post_history, str):
        target_fields.append(post_history)

    for field_text in target_fields:
        matches = DISCOURSE_TOPIC_REGEX.findall(field_text)
        for t_id in matches:
            topic_ids.add(t_id)

    return list(topic_ids)


def fetch_discourse_topic_data(topic_id):
    """Fetch topic details and parse both official polls and soft votes."""
    if not topic_id:
        return [], []

    topic_url = f"https://discuss.python.org/t/{topic_id}.json"
    try:
        resp = requests.get(
            topic_url,
            headers={"User-Agent": "PEP-Vote-Collector/1.0"},
            timeout=10,
        )
        if not resp.ok:
            return [], []

        topic_data = resp.json()
        posts = topic_data.get("post_stream", {}).get("posts", [])

        extracted_polls = []
        extracted_soft_votes = []

        for post in posts:
            post_id = post.get("id")
            raw_text = post.get("cooked", "")
            username = post.get("username", "")
            pseudo_author = pseudonymize_username(username)

            # 1. Parse Official Polls
            polls = post.get("polls", [])
            for poll in polls:
                options_summary = []
                total_votes = 0

                for opt in poll.get("options", []):
                    v_count = opt.get("votes", 0)
                    total_votes += v_count
                    options_summary.append(
                        {
                            "option_id": opt.get("id"),
                            "html_text": opt.get("html"),
                            "votes": v_count,
                        }
                    )

                extracted_polls.append(
                    {
                        "poll_name": poll.get("name", "poll"),
                        "voters_count": poll.get("voters", 0),
                        "total_votes_cast": total_votes,
                        "options": options_summary,
                        "topic_id": topic_id,
                        "post_id": post_id,
                    }
                )

            # 2. Parse Soft Votes (+1, +0, -0, -1, etc.)
            matches = SOFT_VOTE_PATTERN.findall(raw_text)
            if matches:
                unique_ratings = list(dict.fromkeys(matches))
                for rating in unique_ratings:
                    extracted_soft_votes.append(
                        {
                            "vote_id": f"post{post_id}_{rating.replace('/', '_')}",
                            "rating": rating,
                            "author_pseudonym": pseudo_author,
                            "topic_id": topic_id,
                            "post_id": post_id,
                            "post_number": post.get("post_number"),
                            "created_at": post.get("created_at"),
                            "post_url": f"https://discuss.python.org/t/{topic_id}/{post.get('post_number')}",
                        }
                    )

        return extracted_polls, extracted_soft_votes

    except Exception:
        return [], []


# 3. Main Processing Logic
pep_api_index = fetch_pep_index()
print(f"Loaded {len(pep_api_index)} total PEPs from Official PEP API.")

# Clear target collection prior to upsert
pep_voting_summary_col.delete_many({})

bulk_ops = []
processed_peps_count = 0
saved_documents_count = 0
peps_with_polls_count = 0
peps_with_soft_votes_count = 0

print(
    "Processing PEPs since August 2018 (Filtering for documents with polls or soft votes)..."
)

for pep_str, api_meta in pep_api_index.items():
    try:
        pep_num = int(pep_str)
    except ValueError:
        continue

    # Filter for PEPs created or resolved on/after August 2018
    created_dt = parse_date(api_meta.get("created"))
    created_dt = created_dt if created_dt else AUG_2018_CUTOFF

    if created_dt < AUG_2018_CUTOFF:
        continue

    processed_peps_count += 1
    pep_title = api_meta.get("title", "")

    # Get topic IDs from Resolution & Post-History links
    topic_ids = extract_discourse_topic_ids(api_meta)

    pep_polls = []
    pep_soft_votes = []

    for t_id in topic_ids:
        polls, soft_votes = fetch_discourse_topic_data(t_id)
        pep_polls.extend(polls)
        pep_soft_votes.extend(soft_votes)
        time.sleep(0.12)  # Polite rate limit delay

    # CONDITIONAL CHECK: Skip document creation if NO polls AND NO soft votes exist
    if not pep_polls and not pep_soft_votes:
        continue

    saved_documents_count += 1

    # Calculate soft vote tally summary
    tally_breakdown = {}
    for sv in pep_soft_votes:
        r = sv["rating"]
        tally_breakdown[r] = tally_breakdown.get(r, 0) + 1

    if len(pep_polls) > 0:
        peps_with_polls_count += 1
    if len(pep_soft_votes) > 0:
        peps_with_soft_votes_count += 1

    # Consolidated Single Document Schema per PEP
    summary_doc = {
        "_id": f"pep-{pep_num:04d}",
        "pep_number": pep_num,
        "formatted_id": f"pep-{pep_num:04d}",
        "title": pep_title,
        "type": api_meta.get("type"),
        "status": api_meta.get("status"),
        "created": api_meta.get("created"),
        "resolution_url": api_meta.get("resolution"),
        "discourse_topic_ids": topic_ids,
        "polls": {
            "has_polls": len(pep_polls) > 0,
            "poll_count": len(pep_polls),
            "records": pep_polls,
        },
        "soft_votes": {
            "has_soft_votes": len(pep_soft_votes) > 0,
            "total_soft_votes": len(pep_soft_votes),
            "tally_summary": tally_breakdown,
            "records": pep_soft_votes,
        },
        "metadata": {
            "updated_at": datetime.now(timezone.utc).isoformat(),
        },
    }

    bulk_ops.append(
        ReplaceOne({"_id": summary_doc["_id"]}, summary_doc, upsert=True)
    )

# 4. Bulk Write Execution
if bulk_ops:
    print(
        f"Inserting {len(bulk_ops)} qualifying PEP summary documents into '{pep_voting_summary_col.name}'..."
    )
    pep_voting_summary_col.bulk_write(bulk_ops)
else:
    print(
        "No PEPs with soft votes or polls were found matching the criteria."
    )

print("=" * 65)
print(f"Total PEPs evaluated since Aug 2018:   {processed_peps_count}")
print(f"Documents created (Poll or Vote found): {saved_documents_count}")
print(f"PEPs with official polls:              {peps_with_polls_count}")
print(f"PEPs with soft votes:                  {peps_with_soft_votes_count}")
print(f"Collection update complete:             '{pep_voting_summary_col.name}'")

Loaded 741 total PEPs from Official PEP API.
Processing PEPs since August 2018 (Filtering for documents with polls or soft votes)...
Inserting 91 qualifying PEP summary documents into 'pep_voting_summary'...
Total PEPs evaluated since Aug 2018:   272
Documents created (Poll or Vote found): 91
PEPs with official polls:              6
PEPs with soft votes:                  88
Collection update complete:             'pep_voting_summary'


In [4]:
from datetime import datetime, timezone
import json
import re
from pymongo import MongoClient, UpdateOne

# 1. Load Configuration
with open("config.json") as f:
    config = json.load(f)

# 2. Setup MongoDB Connection
mongo_connection_string = config["mongo_uri"]
client = MongoClient(mongo_connection_string, maxPoolSize=10)
db = client.consensus

pep_pr_comments_col = db.pep_pr_comments
pep_voting_summary_col = db.pep_voting_summary

# Regex pattern to capture soft vote ratings (+1, +0, -0, -1, +1/2, -1/2, +0.5, -0.5, etc.)
SOFT_VOTE_PATTERN = re.compile(
    r"(?<![a-zA-Z0-9_])([\+\-](?:1|0|0\.5|1\/2|\.5))(?![a-zA-Z0-9_])"
)

# 3. Query PR Comments with Potential Soft Votes
print(f"Scanning '{pep_pr_comments_col.name}' for PR comment soft votes...")

cursor = pep_pr_comments_col.find(
    {"raw_text": {"$regex": r"[\+\-]"}},
    {
        "url": 1,
        "repo": 1,
        "type": 1,
        "pr_number": 1,
        "pep_numbers": 1,
        "author_id": 1,
        "created_at": 1,
        "raw_text": 1,
        "metadata": 1,
    },
)

# Group extracted PR soft votes by PEP number
pep_pr_votes_map = {}

for comment in cursor:
    raw_text = comment.get("raw_text", "")
    matches = SOFT_VOTE_PATTERN.findall(raw_text)

    if not matches:
        continue

    # Deduplicate ratings within a single comment if repeated (preserving order)
    unique_ratings = list(dict.fromkeys(matches))

    comment_id = comment.get("metadata", {}).get("_id") or comment.get(
        "metadata", {}
    ).get("comment_id")
    pr_num = comment.get("pr_number")
    pep_numbers = comment.get("pep_numbers", [])
    author_id = comment.get("author_id", "anon_unknown")

    for rating in unique_ratings:
        vote_record = {
            "vote_id": f"github_pr{pr_num}_comment{comment_id}_{rating.replace('/', '_')}",
            "rating": rating,
            "author_pseudonym": author_id,
            "pr_number": pr_num,
            "comment_id": comment_id,
            "comment_type": comment.get("type"),
            "url": comment.get("url"),
            "created_at": comment.get("created_at"),
        }

        # Associate vote record with every linked PEP number
        for pep_num in pep_numbers:
            try:
                pep_num_int = int(pep_num)
            except ValueError:
                continue

            if pep_num_int not in pep_pr_votes_map:
                pep_pr_votes_map[pep_num_int] = []

            pep_pr_votes_map[pep_num_int].append(vote_record)

# 4. Prepare Bulk Upsert Operations for pep_voting_summary
bulk_ops = []
total_pr_votes_count = 0

for pep_num, vote_records in pep_pr_votes_map.items():
    formatted_id = f"pep-{pep_num:04d}"
    total_pr_votes_count += len(vote_records)

    # Compute soft vote tally summary breakdown
    tally_breakdown = {}
    for vr in vote_records:
        r = vr["rating"]
        tally_breakdown[r] = tally_breakdown.get(r, 0) + 1

    pr_soft_votes_payload = {
        "has_pr_soft_votes": len(vote_records) > 0,
        "total_pr_soft_votes": len(vote_records),
        "tally_summary": tally_breakdown,
        "records": vote_records,
    }

    # Upsert specific pr_soft_votes field using $set
    bulk_ops.append(
        UpdateOne(
            {"_id": formatted_id},
            {
                "$set": {
                    "pep_number": pep_num,
                    "formatted_id": formatted_id,
                    "pr_soft_votes": pr_soft_votes_payload,
                    "metadata.updated_at": datetime.now(
                        timezone.utc
                    ).isoformat(),
                }
            },
            upsert=True,
        )
    )

# 5. Execute Bulk Write
if bulk_ops:
    print(
        f"Upserting PR soft votes across {len(bulk_ops)} PEP summary documents in '{pep_voting_summary_col.name}'..."
    )
    pep_voting_summary_col.bulk_write(bulk_ops)
else:
    print("No PR comment soft votes were found to upsert.")

print("=" * 65)
print(f"PEPs updated with PR soft votes:      {len(bulk_ops)}")
print(f"Total PR soft vote records mapped:     {total_pr_votes_count}")
print(f"Collection update complete:            '{pep_voting_summary_col.name}'")

Scanning 'pep_pr_comments' for PR comment soft votes...
Upserting PR soft votes across 23 PEP summary documents in 'pep_voting_summary'...
PEPs updated with PR soft votes:      23
Total PR soft vote records mapped:     34
Collection update complete:            'pep_voting_summary'


In [7]:
import json
import pandas as pd
import requests


def fetch_decision_episodes(topic_ids):
    records = []
    headers = {"User-Agent": "DecisionEpisodeExtractor/1.0"}

    for topic_id in topic_ids:
        url = f"https://discuss.python.org/t/{topic_id}.json"
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code != 200:
                print(f"Skipping topic {topic_id}: HTTP {response.status_code}")
                continue

            data = response.json()
            title = data.get("title", "Unknown")
            created_at = data.get("created_at", "")[:10]  # Extract YYYY-MM-DD
            posts = data.get("post_stream", {}).get("posts", [])

            if not posts:
                continue

            # Iterate through all posts in the stream to find embedded polls
            for post in posts:
                polls = post.get("polls", [])
                for poll in polls:
                    voters_count = poll.get("voters", 0)
                    options = poll.get("options", [])

                    # Summarize options into a clean string representation
                    options_breakdown = "; ".join(
                        [
                            f"{opt.get('html', '').strip()}: {opt.get('votes', 0)}"
                            for opt in options
                        ]
                    )

                    records.append(
                        {
                            "Topic ID": topic_id,
                            "Date": created_at,
                            "Title": title,
                            "Poll Name": poll.get("name", "poll"),
                            "Total Voters": voters_count,
                            "Options Breakdown": options_breakdown,
                            "URL": f"https://discuss.python.org/t/{topic_id}",
                        }
                    )
        except Exception as e:
            print(f"Error processing topic {topic_id}: {e}")

    df = pd.DataFrame(records)
    return df


# Verified Discourse Topic IDs from the source map
topic_ids = [2160, 3315, 7010, 10211, 13520, 22901, 23628, 30474, 108706]

print("Fetching decision episodes from Python Discussions...\n")
df_episodes = fetch_decision_episodes(topic_ids)

# 1. Print formatted plain-text table (Native pandas, no tabulate required)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

print("=== Decision Episodes Summary Table ===")
print(df_episodes.to_string(index=False))

# 2. Save directly to CSV for easy inspection/import
csv_filename = "decision_episodes.csv"
df_episodes.to_csv(csv_filename, index=False)
print(f"\nSuccessfully saved {len(df_episodes)} records to '{csv_filename}'.")

Fetching decision episodes from Python Discussions...

Skipping topic 2160: HTTP 404
Skipping topic 3315: HTTP 404
Skipping topic 7010: HTTP 404
Skipping topic 10211: HTTP 404
Skipping topic 13520: HTTP 404
Skipping topic 22901: HTTP 404
=== Decision Episodes Summary Table ===
 Topic ID       Date                                                      Title Poll Name  Total Voters                                                                                                                                                                                                                                                                                                                                                                               Options Breakdown                                 URL
    23628 2023-02-07                         Two polls on how to revise PEP 649     poll2            23                                                                                               